## Preprocess the merged data

In [44]:
%pip install -q -r ../../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [35]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import os

In [45]:
# Load CSV files
data_dir = '../../mcphases/'

merged = pd.read_csv(os.path.join(data_dir, 'merged/physical_activity_merged.csv'))

print("CSV file loaded successfully!")

CSV file loaded successfully!


### 1. Examine missing values

In [46]:
# Number of missing values per column
missing_count = merged.isnull().sum()

# Percentage of missing values per column
missing_percent = ((merged.isnull().sum() / len(merged)) * 100).round(2)

# Combine into a table
missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percent": missing_percent
}).sort_values("Missing Percent", ascending=False)

print(missing_summary)

                                    Missing Count  Missing Percent
pdg                                          3795            67.06
exerciselevel_num                            2329            41.16
fatigue_num                                  2328            41.14
sedentary                                    1961            34.65
sexually_active_num                           372             6.57
estrogen                                      321             5.67
lh                                            320             5.65
filtered_demographic_vo2_max                  285             5.04
cardio_zone                                   209             3.69
peak_zone                                     209             3.69
below_fat_burn_zone                           209             3.69
fat_burn_zone                                 209             3.69
very                                          178             3.15
lightly                                       178             

In [49]:
merged.shape

(5659, 25)

42 participants * 2 periods * 90 days each period = 7560 rows max

In [50]:
merged.duplicated(subset=['id','day_in_study']).sum()

np.int64(0)

### 2. Process missing values

First, fill in the 'study_interval" for time-series interpolation.

In [51]:
merged.groupby(['id', pd.cut(merged['day_in_study'], bins=[0,100,800,1004])])['study_interval'].unique()


C:\Users\caowe\AppData\Local\Temp\ipykernel_8724\1256506962.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  merged.groupby(['id', pd.cut(merged['day_in_study'], bins=[0,100,800,1004])])['study_interval'].unique()


id  day_in_study
1   (0, 100]             [2022.0]
    (100, 800]                NaN
    (800, 1004]               NaN
2   (0, 100]             [2022.0]
    (100, 800]                NaN
                        ...      
49  (100, 800]                NaN
    (800, 1004]               NaN
50  (0, 100]             [2022.0]
    (100, 800]                NaN
    (800, 1004]     [nan, 2024.0]
Name: study_interval, Length: 126, dtype: object

In [52]:
def day_range(day):
    if 1 <= day <= 100:
        return 'range_1'
    elif 800 <= day <= 1010:
        return 'range_2'
    return None

merged['_day_range'] = merged['day_in_study'].apply(day_range)

merged['study_interval'] = (
    merged.groupby(['id', '_day_range'])['study_interval']
    .transform(lambda x: x.ffill().bfill())
)

merged = merged.drop(columns='_day_range')

In [53]:
merged.isnull().sum()

id                                       0
study_interval                           0
is_weekend                               0
day_in_study                             0
sedentary                             1961
lightly                                178
moderately                             178
very                                   178
calories_sum                             4
filtered_demographic_vo2_max           285
filtered_demographic_vo2_max_error     167
peak_zone                              209
cardio_zone                            209
fat_burn_zone                          209
below_fat_burn_zone                    209
phase                                    1
lh                                     320
estrogen                               321
pdg                                   3795
exerciselevel_num                     2329
fatigue_num                           2328
age_of_first_menarche                    0
age                                      0
menstrual_h

Now, we can process other features.

In [54]:
# 1. Exclude features
merged = merged.drop(["sedentary"], axis=1)

In [55]:
# 2. Deal with the flipped time in heart rate zone features: 'peak_zone', 'cardio_zone', 'fat_burn_zone', 'below_fat_burn_zone'
# Study interval 2 reverses study interval 1's column orders
cols = ['peak_zone', 'cardio_zone', 'fat_burn_zone', 'below_fat_burn_zone']
reversed_cols = cols[::-1]  # ['below_fat_burn_zone', 'fat_burn_zone', 'cardio_zone', 'peak_zone']

mask = merged['study_interval'] == 2024.0

merged.loc[mask, cols] = merged.loc[mask, reversed_cols].values

In [56]:
merged[cols]

,peak_zone,cardio_zone,fat_burn_zone,below_fat_burn_zone
0,0.0,0.0,126.0,1036.0
1,5.0,82.0,416.0,512.0
2,5.0,119.0,599.0,368.0
3,0.0,0.0,212.0,613.0
4,8.0,123.0,250.0,308.0
...,...,...,...,...
5654,0.0,0.0,39.0,1285.0
5655,0.0,9.0,65.0,1241.0
5656,0.0,0.0,6.0,1410.0
5657,0.0,0.0,24.0,1409.0


In [57]:
# 3. Impute missing values with median
for col in ["lightly", "moderately", "very", "calories_sum", "filtered_demographic_vo2_max_error",
            'peak_zone', 'cardio_zone', 'fat_burn_zone', 'below_fat_burn_zone', "menstrual_health_literacy_num",
            "exerciselevel_num"]:
    merged[col] = merged[col].fillna(merged[col].median())

In [58]:
# 4. Interpolate missing values for specific features
# First check that there are no duplicate day_in_study values for each id and study_interval combination
merged.groupby(['id', 'study_interval'])['day_in_study'].apply(lambda x: x.duplicated().any()).any()

np.False_

In [59]:
# Now we can safely interpolate the missing values for the specified features
features = ['filtered_demographic_vo2_max', 'lh', 'estrogen']

merged = merged.sort_values(['id', 'study_interval', 'day_in_study'])

merged = merged.set_index('day_in_study')
merged[features] = (
    merged.groupby(['id', 'study_interval'])[features]
    .transform(lambda x: x.interpolate(method='index', limit_direction='both'))
)
merged = merged.reset_index()

"phase" has only 1 missing vlue, let's take a look!

In [60]:
# Find the row
merged[merged['phase'].isna()]

,day_in_study,id,study_interval,is_weekend,lightly,moderately,very,calories_sum,filtered_demographic_vo2_max,filtered_demographic_vo2_max_error,...,phase,lh,estrogen,pdg,exerciselevel_num,fatigue_num,age_of_first_menarche,age,menstrual_health_literacy_num,sexually_active_num
275,6,4,2022.0,False,234.0,34.0,20.0,2288.0,32.57471,0.57274,...,NaN,1.6,241.4,NaN,2.0,0.0,12,24,2.0,0.0


In [61]:
missing_row = merged[merged['phase'].isna()]
id_val = missing_row['id'].values[0]
interval_val = missing_row['study_interval'].values[0]

merged[(merged['id'] == id_val) & (merged['study_interval'] == interval_val)][['day_in_study', 'phase']].sort_values('day_in_study').head(20)

,day_in_study,phase
270,1,Menstrual
271,2,Follicular
272,3,Follicular
273,4,Follicular
274,5,Fertility
275,6,NaN
276,7,Fertility
277,8,Fertility
278,9,Fertility
279,10,Fertility


Obviously, the nan value should be 'Fertility'.

In [62]:
# row 275
merged.loc[275, 'phase'] = 'Fertility'

In [63]:
# 5. Fill in a special value for the feature 'sexually_active_num'
merged['sexually_active_num'] = merged['sexually_active_num'].fillna(-1)

___

In [64]:
# Number of missing values per column
missing_count = merged.isnull().sum()

# Percentage of missing values per column
missing_percent = ((merged.isnull().sum() / len(merged)) * 100).round(2)

# Combine into a table
missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percent": missing_percent
}).sort_values("Missing Percent", ascending=False)

print(missing_summary)

                                    Missing Count  Missing Percent
pdg                                          3795            67.06
fatigue_num                                  2328            41.14
day_in_study                                    0             0.00
id                                              0             0.00
lightly                                         0             0.00
moderately                                      0             0.00
study_interval                                  0             0.00
is_weekend                                      0             0.00
calories_sum                                    0             0.00
very                                            0             0.00
filtered_demographic_vo2_max                    0             0.00
filtered_demographic_vo2_max_error              0             0.00
fat_burn_zone                                   0             0.00
below_fat_burn_zone                             0             

Now, let's handle the 2/3 missingness of pdg!

In [65]:
# Is missingness random, or structured?
merged.groupby('study_interval')['pdg'].apply(lambda x: x.isna().mean())

study_interval
2022.0    1.000000
2024.0    0.049465
Name: pdg, dtype: float64

pdg simply wasn't collected at all during the first study interval, likely because it wasn't part of the study protocol yet, or that assay was added later.

Since pdg is one of the hormones used to predict the phase label for the Mira device, the 'phase' feature include some information about the pdg hormone, to a certian degree. Thus, it's reasonable to preclude this feature.

In [66]:
merged = merged.drop(["pdg"], axis=1)

___

Now, let's handle the missing values in the target variable!

In [67]:
merged_fatigue = merged[merged['fatigue_num'].notnull()]

In [68]:
merged_fatigue.shape

(3331, 23)

In [69]:
# Number of missing values per column
missing_count = merged_fatigue.isnull().sum()

# Percentage of missing values per column
missing_percent = ((merged_fatigue.isnull().sum() / len(merged_fatigue)) * 100).round(2)

# Combine into a table
missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percent": missing_percent
}).sort_values("Missing Percent", ascending=False)

print(missing_summary)

                                    Missing Count  Missing Percent
day_in_study                                    0              0.0
id                                              0              0.0
study_interval                                  0              0.0
is_weekend                                      0              0.0
lightly                                         0              0.0
moderately                                      0              0.0
very                                            0              0.0
calories_sum                                    0              0.0
filtered_demographic_vo2_max                    0              0.0
filtered_demographic_vo2_max_error              0              0.0
peak_zone                                       0              0.0
cardio_zone                                     0              0.0
fat_burn_zone                                   0              0.0
below_fat_burn_zone                             0             

### 3. Look at the distribution of the features

In [70]:
merged_fatigue.columns

Index(['day_in_study', 'id', 'study_interval', 'is_weekend', 'lightly',
       'moderately', 'very', 'calories_sum', 'filtered_demographic_vo2_max',
       'filtered_demographic_vo2_max_error', 'peak_zone', 'cardio_zone',
       'fat_burn_zone', 'below_fat_burn_zone', 'phase', 'lh', 'estrogen',
       'exerciselevel_num', 'fatigue_num', 'age_of_first_menarche', 'age',
       'menstrual_health_literacy_num', 'sexually_active_num'],
      dtype='object')

The feature "study_interval" was only used to perform time interpolation for missing values. Thus, it can be removed now.

In [71]:
merged_fatigue = merged_fatigue.drop(columns=['study_interval'])

In [72]:
merged_fatigue.shape

(3331, 22)

#### Categorical features

'is_weekend', 'phase', 'exerciselevel_num', 'fatigue_num', 'age_of_first_menarche', 'age', 'menstrual_health_literacy_num', 'sexually_active_num'

In [73]:
categorical_features_by_day = ['is_weekend', 'phase', 'exerciselevel_num', 'fatigue_num']
categorical_features_by_participant = ['age_of_first_menarche', 'age', 'menstrual_health_literacy_num', 'sexually_active_num']

In [74]:
# Day-level: counts and proportion
phase_order = ['Menstrual', 'Follicular', 'Fertility', 'Luteal']

for col in categorical_features_by_day:
    print(f"\n=== {col} ===")

    if col == 'phase':
        counts = merged_fatigue[col].value_counts(dropna=False).reindex(phase_order)
        props = merged_fatigue[col].value_counts(normalize=True, dropna=False).reindex(phase_order)
    else:
        counts = merged_fatigue[col].value_counts(dropna=False).sort_index()
        props = merged_fatigue[col].value_counts(normalize=True, dropna=False).sort_index()

    display(pd.DataFrame({'count': counts, 'proportion': props.round(3)}))


# Participant-level: one row per id
participants = merged_fatigue.drop_duplicates(subset='id')

for col in categorical_features_by_participant:
    print(f"\n=== {col} (per participant) ===")
    counts = participants[col].value_counts(dropna=False).sort_index()
    props = participants[col].value_counts(normalize=True, dropna=False).sort_index()
    display(pd.DataFrame({'count': counts, 'proportion': props.round(3)}))


=== is_weekend ===


,count,proportion
is_weekend,,
False,2388,0.717
True,943,0.283



=== phase ===


,count,proportion
phase,,
Menstrual,637,0.191
Follicular,841,0.252
Fertility,735,0.221
Luteal,1118,0.336



=== exerciselevel_num ===


...

,count,proportion
age,,
20,10,0.238
21,7,0.167
22,5,0.119
23,4,0.095
24,5,0.119
25,5,0.119
26,1,0.024
27,3,0.071
29,1,0.024



=== menstrual_health_literacy_num (per participant) ===


,count,proportion
menstrual_health_literacy_num,,
0.0,1,0.024
1.0,4,0.095
2.0,24,0.571
3.0,12,0.286
4.0,1,0.024



=== sexually_active_num (per participant) ===


,count,proportion
sexually_active_num,,
-1.0,3,0.071
0.0,24,0.571
1.0,15,0.357


Since only 6 datapoints (less than 1%) recorded exercise level of 0, it should be merged with similar, in this case, level 1, categories.
The same holds for menstrual_health_literacy_num. There are only 42 participants, so only 1 participant in a category is not representative of the whole distribution.

In [75]:
merged_fatigue['exerciselevel_num'] = merged_fatigue['exerciselevel_num'].replace({0: 1})

In [76]:
merged_fatigue['menstrual_health_literacy_num'] = merged_fatigue['menstrual_health_literacy_num'].replace({0: 1, 4: 3})

#### Numerical features

'lightly', 'moderately', 'very', 'calories_sum', 'filtered_demographic_vo2_max',
'filtered_demographic_vo2_max_error', 'peak_zone', 'cardio_zone',
'fat_burn_zone', 'below_fat_burn_zone', 'lh', 'estrogen'

In [30]:
merged_fatigue.to_csv("../../mcphases/merged/physical_activity_merged_processed.csv", index=False)